# Fine-tune Qwen2.5-3B-Instruct (LoRA) trên UIT-ViQuAD2.0 — tối ưu EM/F1

Notebook extractive QA (SQuAD 2.0) với các tối ưu để **tăng EM/F1**:
- Prompt ép **chỉ trả span** (không câu dài)
- **Post-process**: bóc prefix + align prediction về span trong context khi eval
- LoRA `r=32`, `alpha=64` (3B cần capacity cao hơn 1.5B)

**GPU khuyến nghị:** ≥16GB VRAM (RTX 5060 Ti 15GB: `batch_size=1`, OOM → bật `LOAD_IN_4BIT=True`)

Chạy: pip → Restart Kernel → Cell warnings → Run All

In [ ]:
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

In [ ]:
!pip install transformers trl peft accelerate bitsandbytes xformers datasets --no-cache-dir
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-cache-dir
!pip install unsloth_zoo scikit-learn --no-cache-dir

In [ ]:
import importlib
for pkg in ["torch", "transformers", "datasets", "unsloth"]:
    importlib.import_module(pkg)
    print(f"OK  {pkg}")

In [ ]:
import warnings, logging, os, sys
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"
for _n in ("transformers", "datasets", "torch", "unsloth", "peft", "accelerate", "huggingface_hub"):
    logging.getLogger(_n).setLevel(logging.ERROR)
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass
print("Warnings suppressed.")

In [ ]:
import json
import math
import re
import string
import unicodedata
from collections import Counter
from pathlib import Path

import torch
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# ========== CẤU HÌNH 3B ==========
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DATASET_NAME = "taidng/UIT-ViQuAD2.0"
NO_ANSWER_SENTINEL = "Không có đáp án trong đoạn văn"

# Prompt siết chặt → ép model trả span-only (tăng EM)
SYSTEM_PROMPT = (
    "Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. "
    "QUY TẮC BẮT BUỘC:\n"
    "1) Chỉ trả về ĐÚNG cụm từ xuất hiện trong đoạn văn, không thêm bất kỳ từ nào khác.\n"
    "2) Không viết câu hoàn chỉnh, không giải thích, không thêm 'Đáp án:', 'là', 'thủ đô'.\n"
    f"3) Nếu không có đáp án trong đoạn văn, chỉ trả về: {NO_ANSWER_SENTINEL}"
)

USER_PROMPT_TEMPLATE = (
    "Trích xuất câu trả lời từ đoạn văn dưới đây. "
    "Chỉ trả về cụm từ trong đoạn văn, không thêm từ nào khác.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    f"Câu trả lời (span-only hoặc '{NO_ANSWER_SENTINEL}'):"
)

LORA_SAVE_PATH = "qwen2.5-3b-instruct-lora-viquad2"
EVAL_RESULTS_PATH = "eval_results_viquad2_3b.json"
PROFILING_CONFIG_PATH = "profiling_config_3b.json"

DATASET_ROOT = Path("../../")
DEV_JSON_PATH = DATASET_ROOT / "dev_viquad2.json"
TEST_JSON_PATH = DATASET_ROOT / "test_viquad2.json"

RUN_TRAINING = True
RUN_EVALUATION = True
FORCE_HF_DATASET = True
FORCE_REEXPORT_JSON = True

# 3B trên 15GB: mặc định fp16 LoRA; OOM → LOAD_IN_4BIT=True
LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
MAX_SEQ_CAP = 4096
MIN_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 32  # đáp án extractive thường ngắn
USE_SPAN_POSTPROCESS = True  # align prediction về span trong context khi eval

TQDM_BAR = "{desc}: {percentage:3.0f}%|{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"


def build_messages(context, question, answer=None, for_inference=False):
    user_content = USER_PROMPT_TEMPLATE.format(context=context, question=question)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    if answer is not None and not for_inference:
        messages.append({"role": "assistant", "content": answer})
    return messages


def sample_to_train_text(sample, tok):
    messages = build_messages(sample["context"], sample["question"], sample["answer"])
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def load_tokenizer(model_path=BASE_MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(model_path)
    if tok.chat_template is None:
        raise RuntimeError(f"Tokenizer {model_path} thiếu chat_template.")
    return tok

In [ ]:
def row_to_sample(row):
    is_impossible = bool(row["is_impossible"])
    texts = (row.get("answers") or {}).get("text") or []
    if is_impossible or len(texts) == 0:
        return {
            "id": row.get("id", ""), "title": row.get("title", ""),
            "context": row["context"], "question": row["question"],
            "answer": NO_ANSWER_SENTINEL, "is_impossible": True,
        }
    answer = texts[0]
    if not str(answer).strip():
        return {
            "id": row.get("id", ""), "title": row.get("title", ""),
            "context": row["context"], "question": row["question"],
            "answer": NO_ANSWER_SENTINEL, "is_impossible": True,
        }
    return {
        "id": row.get("id", ""), "title": row.get("title", ""),
        "context": row["context"], "question": row["question"],
        "answer": answer, "is_impossible": False,
    }


def hf_split_to_samples(split_dataset):
    return [row_to_sample(row) for row in split_dataset]


def save_samples_json(samples, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(samples)} → {path.resolve()}")


def split_stats(name, samples):
    n = len(samples)
    n_no = sum(1 for s in samples if s["is_impossible"])
    print(f"{name}: {n} | NoAns: {n_no} ({100*n_no/max(n,1):.1f}%)")


try:
    raw_ds = load_dataset(DATASET_NAME)
    train_samples = hf_split_to_samples(raw_ds["train"])
    dev_samples = hf_split_to_samples(raw_ds["validation"])
    test_samples = hf_split_to_samples(raw_ds["test"])
except Exception as e:
    if FORCE_HF_DATASET:
        raise RuntimeError(f"Không tải được HF: {e}")
    with open(TEST_JSON_PATH, encoding="utf-8") as f: test_samples = json.load(f)
    with open(DEV_JSON_PATH, encoding="utf-8") as f: dev_samples = json.load(f)
    train_samples = dev_samples

split_stats("Train", train_samples)
split_stats("Val", dev_samples)
split_stats("Test", test_samples)
if FORCE_REEXPORT_JSON:
    save_samples_json(dev_samples, DEV_JSON_PATH)
    save_samples_json(test_samples, TEST_JSON_PATH)

In [ ]:
if "train_samples" not in globals():
    raise NameError("Chạy cell tải dataset trước.")


def compute_max_seq_length(samples, tok, cap=MAX_SEQ_CAP, min_len=MIN_SEQ_LENGTH):
    lengths = []
    total = len(samples)
    pbar = tqdm(samples, total=total, desc="Token profiling", unit="sample", bar_format=TQDM_BAR)
    for i, s in enumerate(pbar, 1):
        lengths.append(len(tok.encode(sample_to_train_text(s, tok))))
        pbar.set_postfix(done=f"{i}/{total}")
    lengths.sort()
    n = len(lengths)
    stats = {"min": lengths[0], "p50": lengths[n//2], "p95": lengths[int(n*0.95)],
             "p99": lengths[int(n*0.99)], "max": lengths[-1]}
    chosen = max(((min(math.ceil(stats["p99"]*1.05), cap)+255)//256)*256, min_len)
    stats.update({"chosen_max_seq_length": chosen,
                  "truncated_samples": sum(1 for L in lengths if L > chosen),
                  "truncated_pct": round(100*sum(1 for L in lengths if L > chosen)/n, 3)})
    return chosen, stats


tokenizer_prof = load_tokenizer()
if Path(PROFILING_CONFIG_PATH).exists() and not RUN_TRAINING:
    prof_cfg = json.load(open(PROFILING_CONFIG_PATH, encoding="utf-8"))
    max_seq_length, length_stats = prof_cfg["max_seq_length"], prof_cfg["token_length_stats"]
else:
    max_seq_length, length_stats = compute_max_seq_length(train_samples, tokenizer_prof)
    json.dump({"max_seq_length": max_seq_length, "token_length_stats": length_stats},
              open(PROFILING_CONFIG_PATH, "w", encoding="utf-8"), indent=2)
print(f"max_seq_length = {max_seq_length}")
for k, v in length_stats.items(): print(f"  {k}: {v}")

In [ ]:
if RUN_TRAINING:
    import warnings; warnings.filterwarnings("ignore")
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=LOAD_IN_4BIT,
        load_in_8bit=LOAD_IN_8BIT,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=32, lora_alpha=64,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_dropout=0.05, bias="none",
        use_gradient_checkpointing="unsloth", random_state=3407,
    )
    print("3B + LoRA loaded | quantize:", LOAD_IN_4BIT or LOAD_IN_8BIT)

In [ ]:
if RUN_TRAINING:
    from datasets import Dataset
    def formatting_prompts_func(examples):
        texts = []
        for ctx, q, ans in zip(examples["context"], examples["question"], examples["answer"]):
            msgs = build_messages(ctx, q, ans)
            texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
        return {"text": texts}
    train_hf = Dataset.from_list(train_samples)
    dataset = train_hf.map(formatting_prompts_func, batched=True, remove_columns=train_hf.column_names)
    print(f"Train: {len(dataset)} samples")

In [ ]:
if RUN_TRAINING:
    import warnings; warnings.filterwarnings("ignore")
    from trl import SFTTrainer
    from transformers import TrainingArguments
    from unsloth import is_bfloat16_supported
    import sys

    trainer = SFTTrainer(
        model=model, processing_class=tokenizer, train_dataset=dataset,
        dataset_text_field="text", max_seq_length=max_seq_length,
        dataset_num_proc=1, packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            warmup_steps=50,
            num_train_epochs=2,
            learning_rate=1e-4,
            fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
            logging_steps=20, optim="adamw_8bit", weight_decay=0.01,
            lr_scheduler_type="cosine", seed=3407,
            output_dir="outputs_viquad2_3b", save_strategy="epoch", report_to="none",
        ),
    )
    cfg_cls, tr_cls = type(trainer.args), type(trainer)
    sys.modules["trl.trainer.sft_config"] = sys.modules[cfg_cls.__module__]
    sys.modules["trl.trainer.sft_trainer"] = sys.modules[tr_cls.__module__]
    sys.modules[cfg_cls.__module__].SFTConfig = cfg_cls
    sys.modules[tr_cls.__module__].SFTTrainer = tr_cls
    print("Training 3B...")
    trainer.train()
    model.save_pretrained(LORA_SAVE_PATH)
    tokenizer.save_pretrained(LORA_SAVE_PATH)
    print(f"Saved → {LORA_SAVE_PATH}")

## Evaluation — raw vs span-aligned EM/F1

In [ ]:
PREFIX_RE = re.compile(
    r"^(đáp án|answer|câu trả lời|theo đoạn văn|trong đoạn văn)\s*[:\-]?\s*",
    re.IGNORECASE,
)

def normalize_text(text):
    text = unicodedata.normalize("NFC", text or "")
    return " ".join(text.lower().translate(str.maketrans("", "", string.punctuation)).split())

def compute_em(pred, truth):
    return int(normalize_text(pred) == normalize_text(truth))

def compute_f1(pred, truth):
    pt, tt = normalize_text(pred).split(), normalize_text(truth).split()
    if not pt and not tt: return 1.0
    if not pt or not tt: return 0.0
    common = Counter(pt) & Counter(tt)
    n = sum(common.values())
    if n == 0: return 0.0
    p, r = n/len(pt), n/len(tt)
    return 2*p*r/(p+r)

def is_no_answer(pred):
    return normalize_text(pred) == normalize_text(NO_ANSWER_SENTINEL)

def clean_prediction(raw):
    pred = raw.strip().split("\n")[0].strip().strip('"\'')
    pred = PREFIX_RE.sub("", pred).strip()
    return pred

def align_to_context(pred, context):
    if not pred or is_no_answer(pred):
        return pred
    idx = context.lower().find(pred.lower())
    if idx >= 0:
        return context[idx:idx+len(pred)]
    words, pred_words = context.split(), normalize_text(pred).split()
    if not pred_words:
        return pred
    n = len(pred_words)
    best_span, best_f1 = pred, 0.0
    for i in range(len(words)):
        for j in range(i+1, min(i+n+4, len(words))+1):
            span = " ".join(words[i:j])
            f1 = compute_f1(span, pred)
            if f1 > best_f1:
                best_f1, best_span = f1, span
    return best_span.strip() if best_f1 >= 0.5 else pred

In [ ]:
if RUN_EVALUATION:
    import warnings; warnings.filterwarnings("ignore")
    from transformers import AutoModelForCausalLM
    from peft import PeftModel

    if RUN_TRAINING:
        from unsloth import FastLanguageModel
        model_eval, tokenizer_eval = model, tokenizer
        FastLanguageModel.for_inference(model_eval)
    else:
        tokenizer_eval = load_tokenizer(LORA_SAVE_PATH if Path(LORA_SAVE_PATH).exists() else BASE_MODEL_NAME)
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
        model_eval = PeftModel.from_pretrained(base, LORA_SAVE_PATH).eval()

    model_eval.generation_config.max_length = None
    model_eval.generation_config.max_new_tokens = MAX_NEW_TOKENS

    has_em_r, has_f1_r, has_em_a, has_f1_a, no_ok, preds = [], [], [], [], [], []
    total_eval = len(test_samples)
    pbar = tqdm(test_samples, total=total_eval, desc="Evaluating", unit="sample", bar_format=TQDM_BAR)

    for i, s in enumerate(pbar, 1):
        pbar.set_description(f"Evaluating {i}/{total_eval}")
        msgs = build_messages(s["context"], s["question"], for_inference=True)
        prompt = tokenizer_eval.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer_eval(prompt, return_tensors="pt", truncation=True, max_length=max_seq_length).to(model_eval.device)
        with torch.no_grad():
            out = model_eval.generate(input_ids=inputs["input_ids"], attention_mask=inputs.get("attention_mask"),
                max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                pad_token_id=tokenizer_eval.pad_token_id, eos_token_id=tokenizer_eval.eos_token_id)
        raw = tokenizer_eval.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred_raw = clean_prediction(raw)
        pred = align_to_context(pred_raw, s["context"]) if USE_SPAN_POSTPROCESS else pred_raw

        if s["is_impossible"]:
            ok = int(is_no_answer(pred))
            no_ok.append(ok)
            em_r = em_a = ok; f1_r = f1_a = float(ok)
        else:
            gt = s["answer"]
            em_r = compute_em(pred_raw, gt); f1_r = compute_f1(pred_raw, gt)
            em_a = compute_em(pred, gt); f1_a = compute_f1(pred, gt)
            has_em_r.append(em_r); has_f1_r.append(f1_r)
            has_em_a.append(em_a); has_f1_a.append(f1_a)

        preds.append({"id": s["id"], "question": s["question"], "is_impossible": s["is_impossible"],
            "ground_truth": s["answer"], "prediction_raw": pred_raw, "prediction": pred,
            "em_raw": em_r if not s["is_impossible"] else em_a,
            "f1_raw": f1_r if not s["is_impossible"] else f1_a,
            "em": em_a, "f1": f1_a})
        if has_em_a: pbar.set_postfix(has_em=f"{100*sum(has_em_a)/len(has_em_a):.1f}%")

    n_has, n_no = len(has_em_a), len(no_ok)
    metrics = {
        "hasans_em_raw": round(100*sum(has_em_r)/max(n_has,1), 4),
        "hasans_f1_raw": round(100*sum(has_f1_r)/max(n_has,1), 4),
        "hasans_em": round(100*sum(has_em_a)/max(n_has,1), 4),
        "hasans_f1": round(100*sum(has_f1_a)/max(n_has,1), 4),
        "noans_accuracy": round(100*sum(no_ok)/n_no, 4) if n_no else None,
        "n_hasans": n_has, "n_noans": n_no,
    }

    buckets = {"0-0.2":0,"0.2-0.5":0,"0.5-0.8":0,"0.8-1.0":0,"1.0":0}
    for f1 in has_f1_a:
        if f1==1.0: buckets["1.0"]+=1
        elif f1>=0.8: buckets["0.8-1.0"]+=1
        elif f1>=0.5: buckets["0.5-0.8"]+=1
        elif f1>=0.2: buckets["0.2-0.5"]+=1
        else: buckets["0-0.2"]+=1

    line = "="*55
    print("\n"+line)
    print("   KẾT QUẢ ĐÁNH GIÁ - UIT-ViQuAD2.0 (Qwen2.5-3B + LoRA)")
    print(line)
    print(f"Tổng số câu hỏi test : {total_eval}")
    print(f"HasAns / NoAns      : {n_has} / {n_no}")
    print(f"EM  (raw → aligned) : {metrics['hasans_em_raw']:.2f}% → {metrics['hasans_em']:.2f}%")
    print(f"F1  (raw → aligned) : {metrics['hasans_f1_raw']:.2f}% → {metrics['hasans_f1']:.2f}%")
    if metrics["noans_accuracy"] is not None:
        print(f"NoAns accuracy      : {metrics['noans_accuracy']:.2f}%")
    print(f"max_seq_length      : {max_seq_length}")
    print(line)
    if n_has:
        print(f"\nEM đúng (aligned): {sum(has_em_a)}/{n_has}")
        print("Phân phối F1 (aligned):")
        for b,c in buckets.items(): print(f"  F1={b}: {c} ({100*c/n_has:.1f}%)")

    with open(EVAL_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump({"dataset": DATASET_NAME, "base_model": BASE_MODEL_NAME, "adapter": LORA_SAVE_PATH,
            "max_seq_length": max_seq_length, "use_span_postprocess": USE_SPAN_POSTPROCESS,
            **metrics, "total": len(preds), "predictions": preds}, f, ensure_ascii=False, indent=2)
    print(f"\nSaved → {EVAL_RESULTS_PATH}")